In [ ]:
from __future__ import annotations

import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score

import manifold_dynamics.neural_utils as nu
import manifold_dynamics.paths as pth
import manifold_dynamics.tuning_utils as tut
import visionlab_utils.storage as vst

In [ ]:
# ROI name, should be 3 parts (index.label.category)
target = "19.Unknown.F"
target_parts = target.split(".")
roi_label = f"{int(target_parts[0]):02d}.{target_parts[1]}.{target_parts[2]}"

# load in multi-session ROI data, binned to PSTH
raster_4d = nu.significant_trial_raster(target, alpha=0.05, bin_size_ms=20)

# get top-k value for ROI
topk_local = vst.fetch(f"{pth.OTHERS}/topk_vals.pkl")
with open(topk_local, "rb") as f:
    topk_vals = pickle.load(f)
top_k = int(topk_vals[roi_label]["k"])

print(f"Resolved ROI target: {target}")
print(f"Using top-k = {top_k}")
print(f"Raster shape after binning {raster_4d.shape}")

# shape (units, time, images)
raster_3d = np.nanmean(raster_4d, axis=3)
scores = tut.rank_images_by_response(raster_3d)

idx_topk = scores[:top_k]
trial_3d = raster_3d[:, :, idx_topk]
print(f"Trial averaged, all data {raster_3d.shape}")
print(f"Trial averaged, top-k shape {trial_3d.shape}")

In [ ]:
layer_key = "classifier.2"

# load in AlexNet fc6 embeddings
feature_uri = f"{pth.SAVEDIR}/alexnet/alexnet_acts.pkl"
acts_local = vst.fetch(feature_uri)
with open(acts_local, "rb") as f:
    acts = pickle.load(f)
print(f"Activations available for {list(acts.keys())}")

In [ ]:
layer_acts = acts[layer_key]
layer_acts = layer_acts[1000:]
print(f"Activations for {layer_acts.shape[0]} images")

# normalize fc6 embeddings
scaler = StandardScaler(with_mean=True, with_std=True)
Xz = scaler.fit_transform(layer_acts)

# 60-D PC space (follwing Shi et al)
n_pc = 6
pca = PCA(n_components=n_pc, svd_solver="full")
X_pca = pca.fit_transform(Xz)

# for all 1072 images: 0.498
# for 1000 NSD images: 0.50
# for 72 LOC images: 0.97
print("Total explained variance ratio (60 PCs):", pca.explained_variance_ratio_.sum())

In [ ]:
'''
pretty bad for 60 PCs with 24/24/24 image split

Averaged responses shape: (236, 72)

Processing set1 with 24 images
set1: mean R2 = -0.1961
set1: mean shuffle R2 = -0.4854

Processing set2 with 24 images
set2: mean R2 = -0.1736
set2: mean shuffle R2 = -0.2566

Processing set3 with 24 images
set3: mean R2 = -0.1637
set3: mean shuffle R2 = -0.2127

'''
# average responses over time (100:270)
# raster_3d: (units, time, images)
resp = np.nanmean(raster_3d[:, 100:270, 1000:], axis=1)  # (units, images)
print("Averaged responses shape:", resp.shape)

# transpose to (images, units) to match X
Y = resp.T  # (images, units)

# image sets
face_idx = np.arange(0,24)
body_idx = np.concatenate([np.arange(24,31), np.arange(43,49), np.arange(51,62)])
obj_idx = np.setdiff1d(np.arange(24,72), body_idx)

assert(len(face_idx) == len(body_idx) == len(obj_idx))

sets = {
    "set1": face_idx,
    "set2": body_idx,
    "set3": obj_idx,
}

# Loop over sets and units
alphas = np.logspace(-3, 3, 20)
cv = KFold(n_splits=5, shuffle=True, random_state=0)

results = {}
for set_name, idx in sets.items():
    print(f"\nProcessing {set_name} with {len(idx)} images")

    X_set = X_pca[idx]        # (n_images_subset, 60)
    Y_set = Y[idx]            # (n_images_subset, n_units)

    n_units = Y_set.shape[1]

    r2_all = np.zeros(n_units)
    r2_shuff_all = np.zeros(n_units)
    weights = np.zeros((n_units, n_pc))

    for n in tqdm(range(n_units)):
        y = Y_set[:, n]

        # skip NaN units if any
        if np.all(np.isnan(y)):
            r2_all[n] = np.nan
            r2_shuff_all[n] = np.nan
            continue

        # Fit model (get weights)
        model = RidgeCV(alphas=alphas, cv=5)
        model.fit(X_set, y)
        weights[n] = model.coef_

        # Cross-validated prediction
        model_cv = RidgeCV(alphas=alphas, cv=5)
        y_pred = cross_val_predict(model_cv, X_set, y, cv=cv)
        r2 = r2_score(y, y_pred)
        r2_all[n] = r2

        # Shuffle control (single shuffle for simplicity)
        y_shuff = np.random.permutation(y)
        y_pred_shuff = cross_val_predict(model_cv, X_set, y_shuff, cv=cv)
        r2_shuff = r2_score(y_shuff, y_pred_shuff)
        r2_shuff_all[n] = r2_shuff

    results[set_name] = {
        "r2": r2_all,
        "r2_shuff": r2_shuff_all,
        "weights": weights,
    }

    print(f"{set_name}: mean R2 = {np.nanmean(r2_all):.4f}")
    print(f"{set_name}: mean shuffle R2 = {np.nanmean(r2_shuff_all):.4f}")

In [ ]:
# compare axes across sets
w1 = results["set1"]["weights"]
w2 = results["set3"]["weights"]

cos = np.sum(w1 * w2, axis=1) / (
    np.linalg.norm(w1, axis=1) * np.linalg.norm(w2, axis=1)
)

print("Mean cosine similarity:", np.nanmean(cos))